In [3]:
# ============================================================
# Integral estimation via Sparse Orthogonal Regression (SOR)
# ============================================================
#
# This script generates Figure 3 for the paper:
#   Section 4.3 "Integral estimation"
#
# Purpose
# -------
# The experiment evaluates SOR as an integral estimator using the
# coefficient-readout construction introduced in Section 3.1.
#
# Instead of applying quadrature directly to the target integrand f(x),
# the script:
#
#   1. maps the physical domain [a,b]^d to the canonical domain [-1,1]^d,
#   2. transforms the integrand according to
#
#        g(t) = f(x(t)) / Π_j (1 + t_j^2),
#
#   3. fits a sparse orthonormal expansion of g in a tensor-product basis,
#   4. estimates the integral from the first coefficient c0.
#
# The basis is constructed so that the first basis function is proportional
# to the Riesz representer of the integration functional. With this choice,
# integration reduces to a direct coefficient readout:
#
#     I ≈ ((b-a)/2)^d * c0 * ||1+t^2||^d.
#
# Experimental design
# -------------------
# The script reproduces the four panels of Figure 3:
#
#   (a) 1D Fresnel-type integrals
#       ∫_0^L cos(α x^2) dx
#       evaluated over a range of α.
#
#   (b) 1D Gaussian integrals
#       ∫_0^L exp(-α x^2) dx
#       evaluated over a range of α.
#
#   (c) High-dimensional separable Fresnel-type integrals
#       Π_j cos(α x_j^2)
#       evaluated as a function of dimension d.
#
#   (d) High-dimensional Gaussian integrals
#       exp(-α Σ_j x_j^2)
#       evaluated as a function of dimension d.
#
# In all cases, SOR estimates the integral from pointwise samples only and
# is compared against closed-form reference values on the same finite domain.
#
# Methodological details
# ----------------------
# - The 1D experiments vary α at fixed integration window [0,L].
# - The high-dimensional experiments vary d at fixed α and L.
# - In higher dimensions, the sample count scales linearly with d:
#
#       N = N_base * d,
#
#   so the experiment partially compensates for increasing dimensionality.
#
# - To control basis growth in higher dimensions, the script uses
#   hyperbolic-cross index sets instead of full total-degree sets.
#
# - The transformed target g is optionally normalized before LASSO fitting
#   for numerical stability, and the scaling is undone afterward.
#
# - The LASSO regularization is increased with dimension:
#
#       alpha_eff = lasso_alpha * d,
#
#   to stabilize sparse recovery in higher-dimensional settings.
#
# Connection to the paper
# -----------------------
# This experiment supports the claims in Section 4.3:
#
# - In one dimension, SOR accurately recovers both smooth Gaussian and
#   oscillatory Fresnel-type integrals from samples, without requiring
#   oscillation-resolving quadrature grids.
#
# - In higher dimensions, accuracy degrades gradually with dimension, as
#   expected from sparse approximation theory, but remains useful in
#   moderate dimensions when hyperbolic-cross truncation and dimension-
#   aware regularization are used.
#
# - The experiment demonstrates the main conceptual point of Section 3.1:
#   once a sparse orthogonal expansion of the transformed integrand is
#   recovered, integration becomes a stable linear functional of the
#   learned coefficients.
#
# Outputs
# -------
# The script saves the following figure panels:
#
#   3a.pdf   1D Fresnel-type integral vs α
#   3b.pdf   1D Gaussian integral vs α
#   3c.pdf   High-dimensional Fresnel integral vs d
#   3d.pdf   High-dimensional Gaussian integral vs d
#
# It also saves auxiliary error plots and a small text summary of the
# high-dimensional hyperbolic-cross model sizes.
#
# Notes
# -----
# - The reference values are closed-form values on finite intervals [0,L]
#   or finite boxes [0,L]^d, not improper integrals on unbounded domains.
#
# - The first basis function is modified so that c0 corresponds directly
#   to the integration functional, which is why the estimator differs from
#   a standard Legendre expansion.
#
# - The high-dimensional experiments use separable test integrands so that
#   exact reference values remain available in all tested dimensions.
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso
import mpmath as mp

# ============================================================
# Output directory
# ============================================================
FIGDIR = "figs"
os.makedirs(FIGDIR, exist_ok=True)

# ============================================================
# Global plotting style for all figures
# Adjust these once and reuse everywhere
# ============================================================
PLOT_STYLE = {
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif",
    "axes.labelsize": 16,
    "axes.titlesize": 20,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 12,
    "lines.linewidth": 2.2,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "axes.linewidth": 1.1,
    "savefig.format": "pdf",
}

# Convenient aliases if you want to use them explicitly too
AXIS_LABEL_SIZE = PLOT_STYLE["axes.labelsize"]
TITLE_SIZE = PLOT_STYLE["axes.titlesize"]
TICK_LABEL_SIZE = PLOT_STYLE["xtick.labelsize"]
LEGEND_SIZE = PLOT_STYLE["legend.fontsize"]
LINE_WIDTH = PLOT_STYLE["lines.linewidth"]

plt.rcParams.update(PLOT_STYLE)

# ============================================================
# Map [a,b] -> [-1,1] (componentwise)
# ============================================================
def x_to_t(x, a, b):
    x = np.asarray(x)
    return (2.0 * x - (a + b)) / (b - a)

def t_to_x(t, a, b):
    t = np.asarray(t)
    return 0.5 * ((b - a) * t + (b + a))

# ============================================================
# Canonical 1D orthonormal basis on [-1,1] in variable t
# with phi_0(t) ∝ (1+t^2) so integral is c0 readout.
# ============================================================
NORM_D_SQ = 56.0 / 15.0
NORM_D = float(np.sqrt(NORM_D_SQ))  # ||1+t^2||_{L2([-1,1])}

def phi_1d(t, k):
    t = np.asarray(t)

    if k == 0:
        return np.sqrt(15.0 / 56.0) * (1.0 + t**2)
    elif k == 1:
        return np.sqrt(3.0 / 2.0) * t
    elif k == 2:
        return np.sqrt(21.0 / 2.0) * (5.0 * t**2 - 2.0) / 7.0
    else:
        from numpy.polynomial.legendre import legval
        n = k
        coeffs = np.zeros(n + 1)
        coeffs[-1] = 1.0
        Pn = legval(t, coeffs)
        return np.sqrt((2.0 * n + 1.0) / 2.0) * Pn

def precompute_phi_vals(T, p_max):
    T = np.asarray(T)
    N, d = T.shape
    vals = []
    for j in range(d):
        tj = T[:, j]
        vals_j = [phi_1d(tj, k) for k in range(p_max + 1)]
        vals.append(vals_j)
    return vals

# ============================================================
# Multi-index sets
#   - total-degree (default)
#   - hyperbolic cross (recommended for HD)
# ============================================================
def multi_indices_exact(total_deg, n_vars):
    if n_vars == 1:
        yield (total_deg,)
        return

    def rec(prefix, rem_deg, rem_dims):
        if rem_dims == 1:
            yield tuple(prefix + (rem_deg,))
            return
        for a in range(rem_deg + 1):
            yield from rec(prefix + (a,), rem_deg - a, rem_dims - 1)

    yield from rec((), total_deg, n_vars)

def multi_indices_total_degree(p_max, n_vars):
    idx = []
    for k in range(p_max + 1):
        idx.extend(list(multi_indices_exact(k, n_vars)))
    return idx

def multi_indices_hyperbolic_cross(p_max, n_vars, q=0.75):
    """
    Hyperbolic cross index set:
      {alpha: Π_j (alpha_j + 1)^q <= (p_max + 1)^q }  (equivalently Π_j (alpha_j+1) <= p_max+1 if q=1)
    We also cap each alpha_j <= p_max.
    q in (0,1] typically; smaller q -> stronger sparsity / fewer terms.
    """
    thresh = (p_max + 1) ** q

    idx = []
    def rec(prefix):
        if len(prefix) == n_vars:
            idx.append(tuple(prefix))
            return
        for a in range(p_max + 1):
            new_prefix = prefix + [a]
            prod = 1.0
            for v in new_prefix:
                prod *= (v + 1) ** q
            if prod <= thresh:
                rec(new_prefix)

    rec([])
    idx.sort(key=lambda a: (sum(a), a))
    return idx

# ============================================================
# Tensor-product design matrix using canonical basis
# ============================================================
def build_tensor_design_canonical(T, p_max, indexmulti=None):
    T = np.asarray(T)
    N, d = T.shape
    if indexmulti is None:
        indexmulti = multi_indices_total_degree(p_max, d)
    M = len(indexmulti)

    vals = precompute_phi_vals(T, p_max)

    Phi = np.ones((N, M))
    for s, alpha in enumerate(indexmulti):
        col = np.ones(N)
        for j, deg in enumerate(alpha):
            col *= vals[j][deg]
        Phi[:, s] = col

    return Phi, indexmulti

# ============================================================
# SOR integral estimator on [a,b]^d
# g(t) = f(x(t)) / Π_j (1+t_j^2)
# g ≈ Σ c_alpha Φ_alpha(t)
# I ≈ ((b-a)/2)^d * c_0 * ||1+t^2||^d
#
# Improvements implemented:
#   (i) hyperbolic cross index set in HD
#   (ii) normalize g for stability (and undo afterwards)
#   (iii) allow lasso_alpha to scale with dimension
# ============================================================
def sor_integral_estimate_box(
    X, fX, a, b,
    p_max=4,
    lasso_alpha=2e-4,
    use_hyperbolic_cross=True,
    q_hc=0.75,
    normalize_g=True
):
    X = np.asarray(X)
    fX = np.asarray(fX).reshape(-1)
    N, d = X.shape

    T = x_to_t(X, a, b)
    denom = np.prod(1.0 + T**2, axis=1)
    g = fX / denom

    g_scale = 1.0
    if normalize_g:
        g_scale = float(np.std(g) + 1e-12)
        g = g / g_scale

    if use_hyperbolic_cross and d >= 2:
        indexmulti = multi_indices_hyperbolic_cross(p_max, d, q=q_hc)
    else:
        indexmulti = multi_indices_total_degree(p_max, d)

    Phi, indexmulti = build_tensor_design_canonical(T, p_max, indexmulti=indexmulti)

    # LASSO: scale regularization with dimension to stabilize high-d fits
    alpha_eff = float(lasso_alpha) * float(max(1, d))
    model = Lasso(alpha=alpha_eff, fit_intercept=False, max_iter=400000)
    model.fit(Phi, g)
    c = model.coef_

    # undo scaling
    c = c * g_scale

    c0 = c[0]
    I_hat = ((b - a) / 2.0) ** d * c0 * (NORM_D ** d)

    return float(I_hat), c, indexmulti

# ============================================================
# Closed forms on finite [0,L] (apples-to-apples)
# ============================================================
def true_gaussian_0L(alpha, L):
    alpha = float(alpha)
    return 0.5 * float(mp.sqrt(mp.pi / alpha) * mp.erf(mp.sqrt(alpha) * L))

def true_fresnel_0L(alpha, L):
    alpha = float(alpha)
    z = mp.sqrt(2.0 * alpha / mp.pi) * L
    return float(mp.sqrt(mp.pi / (2.0 * alpha)) * mp.fresnelc(z))

def true_gaussian_d(alpha, L, d):
    return true_gaussian_0L(alpha, L) ** d

def true_fresnel_d(alpha, L, d):
    return true_fresnel_0L(alpha, L) ** d

# ============================================================
# Integrands
# ============================================================
def f_gaussian(X, alpha):
    return np.exp(-alpha * np.sum(X**2, axis=1))

def f_fresnel_separable(X, alpha):
    return np.prod(np.cos(alpha * (X**2)), axis=1)

# ============================================================
# Plot helpers
# ============================================================
def save_curve_with_points(x, y_true, y_hat, xlabel, ylabel, title, fname):
    plt.figure(figsize=(10, 5))
    plt.plot(x, y_true, lw=4, label="Closed form")
    plt.plot(x, y_hat, "o", ms=9, label="SOR estimate")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGDIR, fname), dpi=300)
    plt.close()

def save_error_plot(x, err, xlabel, ylabel, title, fname, logy=True):
    plt.figure(figsize=(10, 5))
    plt.plot(x, err, "o-", lw=2)
    if logy:
        plt.yscale("log")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGDIR, fname), dpi=300)
    plt.close()

# ============================================================
# 1D sweeps over alpha (unchanged conceptually; keep strong)
# ============================================================
def run_1d_sweeps(
    L=8.0,
    N=1600,
    p_max=16,
    lasso_alpha=5e-5,
    alpha_grid=None,
    seed=0
):
    if alpha_grid is None:
        alpha_grid = np.linspace(0.4, 10.0, 22)

    a, b = 0.0, float(L)
    x = np.linspace(a, b, N)
    X = x.reshape(-1, 1)

    # Gaussian 1D
    true_vals = np.array([true_gaussian_0L(al, L) for al in alpha_grid], dtype=float)
    hat_vals = []
    for al in alpha_grid:
        fX = np.exp(-float(al) * x**2)
        Ihat, _, _ = sor_integral_estimate_box(
            X, fX, a, b, p_max=p_max, lasso_alpha=lasso_alpha,
            use_hyperbolic_cross=False, normalize_g=True
        )
        hat_vals.append(Ihat)
    hat_vals = np.array(hat_vals, dtype=float)

    save_curve_with_points(
        alpha_grid, true_vals, hat_vals,
        xlabel=r"Parameter $\alpha$",
        ylabel=r"$I(\alpha)$",
        title=rf"1D Gaussian: $\int_0^L e^{{-\alpha x^2}}dx$ on $[0,L]$, $L={L:g}$",
        fname="3b.pdf"
    )
    save_error_plot(
        alpha_grid, np.abs(hat_vals - true_vals),
        xlabel=r"Parameter $\alpha$",
        ylabel=r"$|\widehat I(\alpha)-I(\alpha)|$",
        title="1D Gaussian: absolute error vs $\\alpha$",
        fname="1d_gaussian_abs_error.png",
        logy=True
    )

    # Fresnel 1D
    true_vals = np.array([true_fresnel_0L(al, L) for al in alpha_grid], dtype=float)
    hat_vals = []
    for al in alpha_grid:
        fX = np.cos(float(al) * x**2)
        Ihat, _, _ = sor_integral_estimate_box(
            X, fX, a, b, p_max=p_max, lasso_alpha=lasso_alpha,
            use_hyperbolic_cross=False, normalize_g=True
        )
        hat_vals.append(Ihat)
    hat_vals = np.array(hat_vals, dtype=float)

    save_curve_with_points(
        alpha_grid, true_vals, hat_vals,
        xlabel=r"Parameter $\alpha$",
        ylabel=r"$I(\alpha)$",
        title=rf"1D Fresnel-type: $\int_0^L \cos(\alpha x^2)\,dx$ on $[0,L]$, $L={L:g}$",
        fname="3a.pdf"
    )
    save_error_plot(
        alpha_grid, np.abs(hat_vals - true_vals),
        xlabel=r"Parameter $\alpha$",
        ylabel=r"$|\widehat I(\alpha)-I(\alpha)|$",
        title="1D Fresnel-type: absolute error vs $\\alpha$",
        fname="1d_fresnel_abs_error.png",
        logy=True
    )

# ============================================================
# Higher-dimensional sweeps over dimension d
#
# Improvements implemented:
#   - sample size grows with d (N ~ base * d)
#   - smaller p_max for HD
#   - hyperbolic cross index set
#   - dimension-scaled LASSO alpha
#   - normalized g
# ============================================================
def run_hd_dimension_sweeps(
    alpha=2.0,
    L=3.0,
    d_list=None,
    N_base=12000,         # samples = N_base * d
    p_max=3,              # lower degree for HD stability
    lasso_alpha=2e-4,
    q_hc=0.7,             # stronger HC sparsity than 0.75
    seed=0
):
    rng = np.random.default_rng(seed)
    if d_list is None:
        d_list = list(range(1, 7))

    a, b = 0.0, float(L)
    d_arr = np.array(d_list, dtype=int)

    # ---- Gaussian in d dims
    true_vals = []
    hat_vals = []
    model_sizes = []
    for d in d_list:
        N = int(N_base * d)  # scale samples with dimension
        X = rng.uniform(a, b, size=(N, d))
        fX = f_gaussian(X, alpha=float(alpha))
        Ihat, c, indexmulti = sor_integral_estimate_box(
            X, fX, a, b,
            p_max=p_max,
            lasso_alpha=lasso_alpha,
            use_hyperbolic_cross=True,
            q_hc=q_hc,
            normalize_g=True
        )
        hat_vals.append(Ihat)
        true_vals.append(true_gaussian_d(alpha, L, d))
        model_sizes.append(len(indexmulti))

    true_vals = np.array(true_vals, dtype=float)
    hat_vals = np.array(hat_vals, dtype=float)

    save_curve_with_points(
        d_arr, true_vals, hat_vals,
        xlabel="Dimension $d$",
        ylabel=r"$I_d(\alpha)$",
        title=rf"Gaussian on $[0,L]^d$: $e^{{-\alpha\sum x_j^2}}$ (fixed $\alpha={alpha:g}$, $L={L:g}$)",
        fname="3d.pdf"
    )
    rel_err = np.abs(hat_vals - true_vals) / np.maximum(np.abs(true_vals), 1e-300)
    save_error_plot(
        d_arr, rel_err,
        xlabel="Dimension $d$",
        ylabel="relative error",
        title="Gaussian: relative error vs dimension",
        fname="hd_gaussian_rel_error.png",
        logy=True
    )

    # ---- Fresnel separable in d dims
    true_vals = []
    hat_vals = []
    model_sizes_f = []
    for d in d_list:
        N = int(N_base * d)
        X = rng.uniform(a, b, size=(N, d))
        fX = f_fresnel_separable(X, alpha=float(alpha))
        Ihat, c, indexmulti = sor_integral_estimate_box(
            X, fX, a, b,
            p_max=p_max,
            lasso_alpha=lasso_alpha,
            use_hyperbolic_cross=True,
            q_hc=q_hc,
            normalize_g=True
        )
        hat_vals.append(Ihat)
        true_vals.append(true_fresnel_d(alpha, L, d))
        model_sizes_f.append(len(indexmulti))

    true_vals = np.array(true_vals, dtype=float)
    hat_vals = np.array(hat_vals, dtype=float)

    save_curve_with_points(
        d_arr, true_vals, hat_vals,
        xlabel="Dimension $d$",
        ylabel=r"$I_d(\alpha)$",
        title=rf"Fresnel separable on $[0,L]^d$: $\prod_j\; \cos(\alpha x_j^2)$ (fixed $\alpha={alpha:g}$, $L={L:g}$)",
        fname="3c.pdf"
    )
    rel_err = np.abs(hat_vals - true_vals) / np.maximum(np.abs(true_vals), 1e-300)
    save_error_plot(
        d_arr, rel_err,
        xlabel="Dimension $d$",
        ylabel="relative error",
        title="Fresnel separable: relative error vs dimension",
        fname="hd_fresnel_rel_error.png",
        logy=True
    )

    # Optional: save a small text summary
    with open(os.path.join(FIGDIR, "hd_summary.txt"), "w") as f:
        f.write(f"alpha={alpha}, L={L}, p_max={p_max}, q_hc={q_hc}, lasso_alpha_base={lasso_alpha}\n")
        f.write("d, N, M(hyperbolic-cross)\n")
        for i, d in enumerate(d_list):
            N = int(N_base * d)
            f.write(f"{d}, {N}, {model_sizes[i]}\n")

# ============================================================
# Run everything + print saved files
# ============================================================
if __name__ == "__main__":
    # 1D sweeps: vary alpha
    run_1d_sweeps(
        L=8.0,
        N=1800,
        p_max=16,
        lasso_alpha=5e-5,
        alpha_grid=np.linspace(0.4, 10.0, 22),
        seed=0
    )

    # HD sweeps: vary d at fixed alpha
    run_hd_dimension_sweeps(
        alpha=2.0,
        L=3.0,
        d_list=list(range(1, 7)),
        N_base=12000,   # N = N_base * d
        p_max=3,
        lasso_alpha=2e-4,
        q_hc=0.7,
        seed=1
    )

    print(f"Saved figures to ./{FIGDIR}/")
    for fn in sorted(os.listdir(FIGDIR)):
        print("  -", fn)

Saved figures to ./figs/
  - 1a.pdf
  - 1b.pdf
  - 1d_fresnel_abs_error.png
  - 1d_gaussian_abs_error.png
  - 2a.pdf
  - 2b.pdf
  - 2c.pdf
  - 2d.pdf
  - 2e.pdf
  - 2f.pdf
  - 3a.pdf
  - 3b.pdf
  - 3c.pdf
  - 3d.pdf
  - 4a.pdf
  - 4b.pdf
  - 4c.pdf
  - 4d.pdf
  - hd_fresnel_rel_error.png
  - hd_gaussian_rel_error.png
  - hd_summary.txt
